In [55]:
import base64
import json
from Crypto.Cipher import AES
from Crypto.Protocol.KDF import PBKDF2
from Crypto.Util.Padding import pad, unpad

class HimkoshEncryptor:
    def __init__(self):
        # Constants from your C# code
        self.password = "Himkosh2015BillServiceAESEncryptDecrypt"
        self.salt = bytes([0x49, 0x76, 0x61, 0x6e, 0x20, 0x4d, 0x65, 0x64, 0x76, 0x65, 0x64, 0x65, 0x76])
        self.key_size = 32 # AES-256
        self.iv_size = 16

    def _derive_key_iv(self):
        # Rfc2898DeriveBytes in C# defaults to 1000 iterations and HMAC-SHA1
        kdf = PBKDF2(self.password, self.salt, dkLen=self.key_size + self.iv_size, count=1000)
        key = kdf[:self.key_size]
        iv = kdf[self.key_size:]
        return key, iv

    def encrypt(self, plain_text):
        key, iv = self._derive_key_iv()
        cipher = AES.new(key, AES.MODE_CBC, iv)
        # C# code uses Encoding.ASCII.GetBytes
        raw_bytes = plain_text.encode('ascii')
        # PKCS7 padding is standard
        padded_data = pad(raw_bytes, AES.block_size)
        encrypted_bytes = cipher.encrypt(padded_data)
        return base64.b64encode(encrypted_bytes).decode('utf-8')
    
    def decrypt(self, encrypted_text):
        # 1. Decode from Base64
        encrypted_bytes = base64.b64decode(encrypted_text)
        
        # 2. Derive the same Key and IV
        key, iv = self._derive_key_iv()
        
        # 3. Setup AES Decryptor
        cipher = AES.new(key, AES.MODE_CBC, iv)
        
        # 4. Decrypt and remove PKCS7 padding
        decrypted_padded = cipher.decrypt(encrypted_bytes)
        decrypted_raw = unpad(decrypted_padded, AES.block_size)
        
        # 5. Convert bytes back to string
        return decrypted_raw.decode('ascii')

# --- Usage ---

# 1. Prepare your data object
data_to_encrypt = {
    "UserName": "eBhugtan",
    "Password": "Himkosh@2025",
    "FinYear": "2019",
    "AccountNo": "55069411800",
    "PayeeName": "san"
}

# 2. Convert to JSON string (compact, no spaces)
json_string = json.dumps(data_to_encrypt, separators=(',', ':'))

# 3. Encrypt
encryptor = HimkoshEncryptor()
new_encrypted_param = encryptor.encrypt(json_string)

print(f"Generated Encrypted Parameter:\n{new_encrypted_param}")


# doen 

Generated Encrypted Parameter:
7JsZWOtl65fM6VjfNVIfnHPQeRaRLQWTbodwGqmPGRLkeAwCes5VCFF9VqdltRATIl5WN3N0+ON6+VTMnzAq1Y1brIpHmxjduX+MiKHf3d4xtCJkX/p5JjF/mCv1iJHkQIL9poaha2YBr6aObdht7g==


In [56]:

import json, requests
raw_res = ""
encrypted_param = "7JsZWOtl65fM6VjfNVIfnHPQeRaRLQWTbodwGqmPGRLkeAwCes5VCFF9VqdltRATIl5WN3N0+ON6+VTMnzAq1Y1brIpHmxjduX+MiKHf3d4xtCJkX/p5JjF/mCv1iJHkQIL9poaha2YBr6aObdht7g=="
req = requests.get(f"http://10.146.15.95/eHPOLTIS/eBhugtanService/eBhugtan.svc/GetBillDetails?param={encrypted_param}")
if req.status_code == 200:
    raw_res = req.text

    print(req.text)
else:
    print(f"Request failed with status code {req.status_code}: {req.text}")

xqxlyQebAApT/i+fDVris2XTBPOLAKkhV6kErLamx/R6nNI1o2MIleVks/23pY/WYvPuvaNjoKWe9KEjm60cUNLDy2V4XBTltG5RHHtFnV9pnMlAVSCmQxyMsWNHGsvfJvdqDQ5GtwbbwzYNHUMqxEFkeSnHGLsMaSYLFaq3oCP84oSfSRXMPQDA5/WJER3tGVkm6CtJ7NbKoM0+DjKaIdH2Lt8TAjUa81heHxXExSWExqG9lB3dwuVpI/3iaRI2kpcbmCQArgx/LlvRH6HDZBDtCIdTLTj/UmInPPvBaFd2y6seehjbeo7BwmVKIHSHdVMEgU6y1QR+vhoW/PwC6bz4uRkZFU6LVykA94xFe+4DxaxHJvXbVv7QPOdmX67+zz19piw4rf5X4oBxfgWq+cHtwJvEXFMbsKoAkWaG/sZYBIY9qr9kRkUh7kr0tIWo7MAD9HyUhnn8K7OMwxKk/IGMnY/VKqiB5ZMpP78M3i/XahkJqPYJXErmBYpvzd/Qb1O8zTlOaOMtKAY70Y7geq518UKq0izIokXR/EKVoCM3KKZ0dCwVVEUHJ9S2ZhTAUg5arOOxuGo0pq//iDDv73UY7OPa/jORwmz3yAhcNUUvKeq/zr1S5g94UupGXT2gS96708b/aN79V16cmpkp0CV2bgXHld2iKsHIDFBxEWvc4lsSbdGAhuXtTJzlOSPkyTh4NabR1eSoZC5vULKSRF430NAb/fASIISilwDDO4lKRebWl78jqMJz15rquLZD2fjokL/Fz83XdzFDTMLf42jihad1OYBuzAThBZGeZmZ6f6k/gwxw8r7QoYEdN4B2bh540/WY0DJyxEIyMsNS+6q0PXEhbikqN/TZJSUmZPiK3JmT+nq7jQhS4PUoAE29TlUoBIFxmQ3+TDoMImihV81CrmaVuaVITS13s4xiURi2NFGiVUkmgv/dA2hXTM5rFCGT1X71JSROQ1baqNDgK6N+GMoCNRsnTrOKMGvY

In [57]:
# decrpt the res
import base64
import json
from Crypto.Cipher import AES
from Crypto.Protocol.KDF import PBKDF2
from Crypto.Util.Padding import pad, unpad

class HimkoshEncryptor:
    def __init__(self):
        # Constants from your C# code
        self.password = "Himkosh2015BillServiceAESEncryptDecrypt"
        self.salt = bytes([0x49, 0x76, 0x61, 0x6e, 0x20, 0x4d, 0x65, 0x64, 0x76, 0x65, 0x64, 0x65, 0x76])
        self.key_size = 32 # AES-256
        self.iv_size = 16

    def _derive_key_iv(self):
        # Rfc2898DeriveBytes in C# defaults to 1000 iterations and HMAC-SHA1
        kdf = PBKDF2(self.password, self.salt, dkLen=self.key_size + self.iv_size, count=1000)
        key = kdf[:self.key_size]
        iv = kdf[self.key_size:]
        return key, iv

    def encrypt(self, plain_text):
        key, iv = self._derive_key_iv()
        cipher = AES.new(key, AES.MODE_CBC, iv)
        # C# code uses Encoding.ASCII.GetBytes
        raw_bytes = plain_text.encode('ascii')
        # PKCS7 padding is standard
        padded_data = pad(raw_bytes, AES.block_size)
        encrypted_bytes = cipher.encrypt(padded_data)
        return base64.b64encode(encrypted_bytes).decode('utf-8')
    
    def decrypt(self, encrypted_text):
        # 1. Decode from Base64
        encrypted_bytes = base64.b64decode(encrypted_text)
        
        # 2. Derive the same Key and IV
        key, iv = self._derive_key_iv()
        
        # 3. Setup AES Decryptor
        cipher = AES.new(key, AES.MODE_CBC, iv)
        
        # 4. Decrypt and remove PKCS7 padding
        decrypted_padded = cipher.decrypt(encrypted_bytes)
        decrypted_raw = unpad(decrypted_padded, AES.block_size)
        
        # 5. Convert bytes back to string
        return decrypted_raw.decode('ascii')



# 3. Decrpty
encryptor = HimkoshEncryptor()

print("--- Decrypting Sample ---")
try:
    decoded_json = encryptor.decrypt(raw_res)
    print(f"Decrypted Content: {decoded_json}")
    
    # Verify by re-encrypting
    re_encrypted = encryptor.encrypt(decoded_json)

except Exception as e:
    print(f"Decryption failed: {e}")
# doen 

--- Decrypting Sample ---
Decrypted Content: {"statuscd":"0","message":"Success","data":[{"BillID":"CTO000162019100382","Treasury":"CTO00 - CAPITAL TREASURY","DDOCode":"016 - D.D.(HIPA) FAIR LAWNS SHIMLA","PayeeCode":"T001610043","PayeeName":"Sandeep Sood","BillType":"TRAINING","Remarks":"Training bill","NetAmount":600.00,"GrossAmount":600.00,"PaidON":"19-12-2019"},{"BillID":"CTO000162019100332","Treasury":"CTO00 - CAPITAL TREASURY","DDOCode":"016 - D.D.(HIPA) FAIR LAWNS SHIMLA","PayeeCode":"T001610043","PayeeName":"Sandeep Sood","BillType":"TRAINING","Remarks":"Training bill","NetAmount":1200.00,"GrossAmount":1200.00,"PaidON":"18-11-2019"},{"BillID":"CTO000162019100422","Treasury":"CTO00 - CAPITAL TREASURY","DDOCode":"016 - D.D.(HIPA) FAIR LAWNS SHIMLA","PayeeCode":"T001610043","PayeeName":"Sandeep Sood","BillType":"TRAINING","Remarks":"Training bill","NetAmount":1200.00,"GrossAmount":1200.00,"PaidON":"20-01-2020"}]}


In [ ]:
Decrypted Content: {"statuscd":"0","message":"Success","data":[{"BillID":"CTO0001
Decrypted Content: {"statuscd":"1","message":"Payee Not registered in IFMIS.","data":[]}
Decrypted Content: {"statuscd":"2","message":"No Record Found.","data":[]}
Decrypted Content: {"statuscd":"2","message":"User not Found.","data":[]}